# 05 · Feature importance

Goal: understand which features the final model relies on. Expected:
area and district matter most; the VIP flags matter little; rooms may be
largely redundant with area. Also: why "repair" can lower the predicted
price for some apartments.

HistGradientBoosting has no built-in `feature_importances_`, so
**permutation importance** is used: shuffle one feature on unseen data
and measure how much MAE gets worse. The bigger the increase, the more
the model depends on that feature.

## 1. Reproduce the model

`models/model.pkl` is trained on all data, so there is no unseen data
left to measure it on. The same model is retrained here on the same 80%
dev split as in `04_model_selection`, and checked on the 20% hold-out.
Hold-out MAE should match 04 (~37,456).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("../data/item_clean_all.csv")
df.shape

(44382, 13)

In [2]:
def prepare_features(df):
    """Same feature preparation as in 02 and 04: group rare districts,
    one-hot encode, drop non-features, fill missing hasRepair."""
    df = df.copy()
    counts = df["location"].value_counts()
    common = counts[counts >= 5].index
    df["location"] = df["location"].where(df["location"].isin(common), "Other")
    df = pd.get_dummies(df, columns=["location"])

    features = df.drop(columns=["id", "price", "currency", "price_per_m2", "city"])
    features["hasRepair"] = features["hasRepair"].fillna(features["hasRepair"].mode()[0])
    target = df["price"]
    return features, target

In [3]:
features, target = prepare_features(df)
X = features.astype(float)
y = target

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_dev.shape, X_holdout.shape

((35505, 90), (8877, 90))

In [4]:
best_params = {
    "learning_rate": 0.03776689505815151,
    "max_iter": 1100,
    "max_leaf_nodes": 222,
    "min_samples_leaf": 5,
    "l2_regularization": 0.0018323505683299347,
    "max_features": 0.5371847150398951,
}

model = HistGradientBoostingRegressor(
    early_stopping=False, random_state=42, **best_params
)
model.fit(X_dev, y_dev)

mae = mean_absolute_error(y_holdout, model.predict(X_holdout))
print(f"hold-out MAE: {mae:,.0f}")

hold-out MAE: 37,456


## 2. Permutation importance

For each feature, its values are shuffled across the hold-out apartments
(5 times), and the increase in MAE is measured. The result is in AZN:
"how much worse the model gets without this feature".

In [5]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_holdout, y_holdout,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X_holdout.columns,
    "mae_increase": result.importances_mean,
    "std": result.importances_std,
}).sort_values("mae_increase", ascending=False)

importance.head(15)

,feature,mae_increase,std
1,area,95613.496211,650.061528
3,floors,24081.280136,450.237638
2,floor,11773.195720,339.401342
0,rooms,9642.800237,182.581920
18,location_Ağ şəhər,8413.624000,172.041455
4,hasRepair,5662.093019,151.508246
43,location_Masazır,4414.170070,35.271555
9,location_28 May,3504.063508,123.828318
32,location_Elmlər Akademiyası,3258.671871,68.231061
69,location_Sea Breeze,2705.524615,91.371739


**Findings:** Area dominates (shuffling it adds ~95.6k to MAE). Total
floors (24.1k) matter twice as much as the apartment's own floor
(11.8k), consistent with EDA: building height is a proxy for building
type and age. Rooms still matter (9.6k): at a given area, room count
says something about the layout (old vs new building). Note: permuting
one of two correlated features (rooms/area) creates unrealistic
combinations, which can inflate its importance. Ağ şəhər is the most
important single district. The VIP flags are not in the top 15.

## 3. District as one feature

The 83 `location_*` columns are one feature split into pieces. Here they
are shuffled together (each apartment gets another apartment's whole
district), so district can be compared fairly with area.

In [6]:
import numpy as np

location_cols = [c for c in X_holdout.columns if c.startswith("location_")]
groups = {c: [c] for c in X_holdout.columns if not c.startswith("location_")}
groups["location (all districts)"] = location_cols

baseline = mean_absolute_error(y_holdout, model.predict(X_holdout))
rng = np.random.default_rng(42)

rows = []
for name, cols in groups.items():
    increases = []
    for _ in range(5):
        X_perm = X_holdout.copy()
        order = rng.permutation(len(X_perm))
        X_perm[cols] = X_holdout[cols].to_numpy()[order]
        mae_perm = mean_absolute_error(y_holdout, model.predict(X_perm))
        increases.append(mae_perm - baseline)
    rows.append({
        "feature": name,
        "mae_increase": np.mean(increases),
        "std": np.std(increases),
    })

grouped = pd.DataFrame(rows).sort_values("mae_increase", ascending=False)
grouped

,feature,mae_increase,std
1,area,95995.201105,1019.753675
7,location (all districts),49917.104979,761.264179
3,floors,23871.200072,137.925599
2,floor,11727.662770,215.219703
0,rooms,9592.465788,121.160567
4,hasRepair,5493.044039,283.198696
6,isFeatured,182.547346,19.548895
5,isVipped,166.732495,52.941859


**Findings:** Shuffled together, the 83 district columns are the second
most important feature (+49.9k MAE), far ahead of anything except area
(+96.0k). One by one they looked minor (max 8.4k), which is why grouping
matters for one-hot features. The custom implementation matches
sklearn's `permutation_importance` for the single features. The VIP flags
add only ~170 AZN (<0.5% of MAE): fixing them in the API costs almost
nothing, and they could be dropped from the model entirely.
Summary: price is driven by area and district, then building type
(floors) and layout (rooms).

## 4. What-if: repair

Every hold-out apartment is predicted twice, with `hasRepair` = 0 and 1,
all other features unchanged. The difference is how much the model
thinks repair adds to that specific apartment.

In [7]:
X_no_repair = X_holdout.copy()
X_no_repair["hasRepair"] = 0.0

X_repair = X_holdout.copy()
X_repair["hasRepair"] = 1.0

repair_effect = pd.Series(
    model.predict(X_repair) - model.predict(X_no_repair),
    index=X_holdout.index,
)

print(repair_effect.describe())
print("share where repair LOWERS the price:", round((repair_effect < 0).mean() * 100, 1), "%")

count    8.877000e+03
mean     2.197302e+04
std      5.865365e+04
min     -1.496078e+06
25%      9.493404e+03
50%      2.252317e+04
75%      3.713733e+04
max      7.354868e+05
dtype: float64
share where repair LOWERS the price: 15.9 %


**Findings:** For a typical apartment, repair adds ~22.5k AZN (median;
middle half: +9.5k to +37k), about +9% of a median price. Repair raises
the predicted price for 84% of hold-out apartments and lowers it for
15.9%. So the model mostly learned a positive repair effect: using
district, floors and area together, it partly separates repair from
new-build status, which the raw EDA comparison could not. A few extreme
values (−1.5M to +0.7M) come from rare, very large listings where the
model is unreliable.

In [8]:
holdout_rows = df.loc[X_holdout.index]
negative = repair_effect < 0

summary = (
    holdout_rows.assign(repair_lowers_price=negative)
    .groupby("repair_lowers_price")[["area", "floors", "price", "price_per_m2"]]
    .median()
)
print(summary)
print()
print(holdout_rows[negative]["location"].value_counts().head(8))

                      area  floors     price  price_per_m2
repair_lowers_price                                       
False                 83.0    15.0  232500.0       2819.83
True                 110.0    14.0  308000.0       2923.08

location
Nəsimi                231
Nizami                176
Elmlər Akademiyası    125
İnşaatçılar           108
Xətai                  89
Şah İsmayıl Xətai      79
Sahil                  70
Gənclik                61
Name: count, dtype: int64


**Findings:** Apartments where repair lowers the prediction are larger
(median 110 vs 83 m²) and more expensive (308k vs 233k), but not in
taller buildings (14 vs 15 floors). The most frequent districts are
central ones (Nəsimi, Nizami, Elmlər Akademiyası...), not the new premium
complexes expected from EDA, so that hypothesis is only half confirmed.
Raw counts mostly reflect district size, so the share per district is
checked below.

In [9]:
share_by_district = (
    negative.groupby(holdout_rows["location"])
    .agg(["mean", "count"])
    .query("count >= 50")
    .sort_values("mean", ascending=False)
)
share_by_district["mean"] = (share_by_district["mean"] * 100).round(1)
share_by_district.head(10)

,mean,count
location,,
Sahil,81.4,86
Nizami,64.9,271
Nəsimi,52.3,442
Xətai,47.6,187
Yeni Yasamal,33.1,139
8 Noyabr,30.7,153
Elmlər Akademiyası,28.3,441
İnşaatçılar,23.4,461
Şah İsmayıl Xətai,23.0,344


**Findings:** The share of apartments where repair lowers the prediction
is highest in the historic centre: Sahil 81%, Nizami 65%, Nəsimi 52%,
Xətai 48% (vs 15.9% overall), all with 86–442 hold-out apartments, so
not noise. The odd Xətai example was typical for that district, not a
one-off. Likely reason: the centre mixes old resale buildings (usually
already renovated) with expensive new buildings sold without finishing,
so there "no repair" mostly means "new and expensive". The data has no
new-build flag, so this cannot be checked directly, but every result
here is consistent with it.

## Summary

- Price is driven by area (+96k MAE when shuffled) and district (+50k),
  then building height (+24k), floor (+12k) and rooms (+10k).
- The VIP flags are almost irrelevant (~170 AZN) and could be removed.
- Repair adds ~22.5k AZN for a typical apartment, but in central
  districts the model confuses "no repair" with "new building".
- Next improvement: scrape the new-build/resale flag from bina.az.